# Subtask 1 — Ensemble Analysis

Objectives:
- Identify patches that are **consistently wrong** across all runs (hard cases / structural anomalies).
- Measure **run diversity**: pairwise prediction disagreement, per-class recall spread.
- Evaluate **ensemble strategies**: mean probs, weighted probs, majority vote.
- Produce a ranked candidate list for the final submission ensemble.

**Label-space note**: pre-fix runs (`existing_*`) trained with wrong label mapping.
Only `corrected_*` runs are used for ensemble experiments. Pre-fix runs are loaded
separately for reference comparison only.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / 'scripts').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Run from the ai4agri repo root')

VAL_PRED_DIR = REPO_ROOT / 'results' / 'subtask1' / 'val_preds'
VISION_RUNS_DIR = REPO_ROOT / 'results' / 'subtask1' / 'vision_runs'

# ── config ──────────────────────────────────────────────────────────────────
# Anomaly threshold: patch is "hard" when Acc±1 < this in a given run
ANOMALY_THRESHOLD = 0.50
# A patch is flagged as anomalous when at least this fraction of runs call it hard
ANOMALY_MIN_RUN_FRACTION = 0.75
# Number of anomalous patches to display visually
DISPLAY_N = 12
# Pre-fix runs included for reference diversity comparison (not for ensemble)
REFERENCE_RUN_IDS: list[str] = [
    'existing_tiny_vit_softce_expected_summary_full_e12_m3_s64',
    'existing_resnet_fpn_softce_expected_summary_full_e18_m3_s63',
]

CLASS_NAMES = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
CLASS_COLORS = ['#d73027', '#fc8d59', '#fee08b', '#91cf60', '#1a9850']
NODATA_COLOR = '#808080'

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['font.size'] = 9
print('Repo root:', REPO_ROOT)

## 1. Discover runs

In [ ]:
def npz_to_run_id(path: Path) -> str:
    return path.stem.replace('_val_probs', '')

all_npz = sorted(VAL_PRED_DIR.glob('*_val_probs.npz'))
corrected_npz = [p for p in all_npz if npz_to_run_id(p).startswith('corrected_')]
reference_npz = [p for p in all_npz if npz_to_run_id(p) in REFERENCE_RUN_IDS]

print(f'Total NPZ files: {len(all_npz)}')
print(f'Corrected runs (for ensemble): {len(corrected_npz)}')
for p in corrected_npz:
    size_mb = p.stat().st_size / 1e6
    print(f'  {npz_to_run_id(p)}  ({size_mb:.0f} MB)')
print(f'Reference runs (pre-fix, for comparison): {len(reference_npz)}')
for p in reference_npz:
    size_mb = p.stat().st_size / 1e6
    print(f'  {npz_to_run_id(p)}  ({size_mb:.0f} MB)')

## 2. Load corrected runs and compute per-patch metrics

In [ ]:
def load_run(path: Path) -> dict:
    p = np.load(path)
    return {
        'run_id': npz_to_run_id(path),
        'patch_ids': p['patch_ids'].astype(str),
        'probs': p['probs'].astype(np.float32),   # (N, 5, H, W)
        'y_true': p['y_true'].astype(np.int16),    # (N, H, W) model 0-4 / nodata 255
        'y_pred': p['y_pred'].astype(np.int16),    # (N, H, W)
    }

def patch_acc_pm1(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Per-patch Acc±1. Returns shape (N,)."""
    valid = (y_true >= 0) & (y_true <= 4)
    diff = np.abs(y_true.astype(np.int16) - y_pred.astype(np.int16))
    pm1 = (diff <= 1) & valid
    n_valid = valid.sum(axis=(1, 2))
    return np.where(n_valid > 0, pm1.sum(axis=(1, 2)) / n_valid, np.nan)

def global_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    valid = (y_true >= 0) & (y_true <= 4)
    diff = np.abs(y_true[valid].astype(np.int16) - y_pred[valid].astype(np.int16))
    acc_pm1 = float((diff <= 1).mean()) if valid.any() else np.nan
    exact = float((diff == 0).mean()) if valid.any() else np.nan
    mae = float(diff.mean()) if valid.any() else np.nan
    per_class = {}
    for c in range(5):
        mask = valid & (y_true == c)
        if mask.any():
            per_class[c] = float((y_pred[mask] == c).mean())
        else:
            per_class[c] = np.nan
    return {'acc_pm1': acc_pm1, 'exact': exact, 'mae': mae, 'per_class_recall': per_class}

print('Loading corrected runs...')
runs: list[dict] = []
for npz_path in corrected_npz:
    run = load_run(npz_path)
    run['patch_pm1'] = patch_acc_pm1(run['y_true'], run['y_pred'])
    run['metrics'] = global_metrics(run['y_true'], run['y_pred'])
    runs.append(run)
    print(f"  {run['run_id']}: Acc±1={run['metrics']['acc_pm1']:.4f}, patches={len(run['patch_ids'])}")

# Canonical patch_ids from the run with most patches
canonical_run = max(runs, key=lambda r: len(r['patch_ids']))
canonical_ids = canonical_run['patch_ids']
print(f"\nCanonical patch set: {len(canonical_ids)} patches from {canonical_run['run_id']}")

## 3. Per-run summary

In [ ]:
rows = []
for r in runs:
    m = r['metrics']
    row = {
        'run_id': r['run_id'].replace('corrected_unet_', '').replace('corrected_', ''),
        'acc_pm1': m['acc_pm1'],
        'exact': m['exact'],
        'mae': m['mae'],
        'patches': len(r['patch_ids']),
    }
    for c, name in enumerate(CLASS_NAMES):
        row[f'recall_{name}'] = m['per_class_recall'].get(c, np.nan)
    rows.append(row)

summary_df = pd.DataFrame(rows).sort_values('acc_pm1', ascending=False).reset_index(drop=True)
display(summary_df.style.format({
    'acc_pm1': '{:.4f}', 'exact': '{:.4f}', 'mae': '{:.4f}',
    **{f'recall_{n}': '{:.3f}' for n in CLASS_NAMES}
}).background_gradient(subset=['acc_pm1'], cmap='RdYlGn'))

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5), sharey=True)
short_labels = [r['run_id'].split('_')[3] + '/' + r['run_id'].split('_')[-1] for r in runs]
for idx, (c, name) in enumerate(zip(range(5), CLASS_NAMES)):
    recalls = [r['metrics']['per_class_recall'].get(c, np.nan) for r in runs]
    axes[idx].barh(range(len(runs)), recalls, color=CLASS_COLORS[c])
    axes[idx].set_title(name)
    axes[idx].set_xlim(0, 1)
    axes[idx].axvline(0.5, color='black', lw=0.7, ls='--')
    if idx == 0:
        axes[idx].set_yticks(range(len(runs)))
        axes[idx].set_yticklabels(short_labels, fontsize=7)
plt.suptitle('Per-class recall by run', y=1.02)
plt.tight_layout()
plt.show()

## 4. Align runs on shared patches

In [ ]:
# Build a shared patch index: only patches present in ALL corrected runs
id_sets = [set(r['patch_ids']) for r in runs]
shared_ids = sorted(id_sets[0].intersection(*id_sets[1:]) if len(id_sets) > 1 else id_sets[0])
print(f'Shared patches across all {len(runs)} corrected runs: {len(shared_ids)}')

def reindex(run: dict, ids: list[str]) -> dict:
    """Reorder run arrays to match the given patch_id list."""
    idx_map = {pid: i for i, pid in enumerate(run['patch_ids'])}
    order = np.array([idx_map[pid] for pid in ids])
    return {
        'run_id': run['run_id'],
        'probs': run['probs'][order],
        'y_true': run['y_true'][order],
        'y_pred': run['y_pred'][order],
        'patch_pm1': run['patch_pm1'][order],
        'metrics': run['metrics'],
    }

aligned = [reindex(r, shared_ids) for r in runs]
shared_ids_arr = np.array(shared_ids)

# Canonical y_true: take from the run with best Acc±1 (they should agree — just a sanity check)
best_run = max(aligned, key=lambda r: r['metrics']['acc_pm1'])
y_true_canonical = best_run['y_true']   # (N, H, W)

# Stack per-patch Acc±1 across runs: shape (n_runs, N)
pm1_matrix = np.stack([r['patch_pm1'] for r in aligned], axis=0)  # (n_runs, N)
print(f'pm1_matrix shape: {pm1_matrix.shape}  — rows=runs, cols=patches')

# Verify y_true is consistent across corrected runs (spot check)
if len(aligned) >= 2:
    match = (aligned[0]['y_true'] == aligned[1]['y_true']).mean()
    print(f'y_true pixel agreement between first two corrected runs: {match:.4f} (expect ~1.0 for same data)')

## 5. Patch anomaly analysis — consistently wrong patches

In [ ]:
# For each patch: fraction of runs calling it hard (Acc±1 < threshold)
hard_matrix = pm1_matrix < ANOMALY_THRESHOLD  # (n_runs, N) bool
hard_fraction = hard_matrix.mean(axis=0)       # (N,) fraction of runs that find it hard
mean_pm1 = np.nanmean(pm1_matrix, axis=0)      # (N,) mean Acc±1 across runs

anomaly_mask = hard_fraction >= ANOMALY_MIN_RUN_FRACTION
print(f'Anomalous patches (hard in ≥{ANOMALY_MIN_RUN_FRACTION:.0%} of runs): {anomaly_mask.sum()} / {len(shared_ids)}')

# Sort anomalous patches by mean Acc±1 ascending (worst first)
anomaly_indices = np.where(anomaly_mask)[0]
anomaly_indices = anomaly_indices[np.argsort(mean_pm1[anomaly_indices])]

# Also note patches where some runs succeed (ensemble opportunity)
# These are patches hard in SOME but not ALL runs
partial_mask = (hard_fraction > 0) & (hard_fraction < 1.0)
print(f'Partially-hard patches (ensemble can help): {partial_mask.sum()}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(mean_pm1, bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].axvline(ANOMALY_THRESHOLD, color='red', ls='--', label=f'threshold {ANOMALY_THRESHOLD}')
axes[0].set_xlabel('Mean Acc±1 across runs')
axes[0].set_ylabel('Patches')
axes[0].set_title('Patch-level Acc±1 distribution')
axes[0].legend()

axes[1].scatter(mean_pm1, hard_fraction, s=4, alpha=0.4, c='steelblue')
if anomaly_mask.any():
    axes[1].scatter(mean_pm1[anomaly_mask], hard_fraction[anomaly_mask], s=8, color='red', alpha=0.7, label='anomaly')
axes[1].set_xlabel('Mean Acc±1 across runs')
axes[1].set_ylabel('Fraction of runs calling it hard')
axes[1].set_title('Hard fraction vs mean Acc±1')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Visual panels — worst anomalous patches

In [ ]:
if len(anomaly_indices) == 0:
    print('No anomalous patches found — lower ANOMALY_MIN_RUN_FRACTION or ANOMALY_THRESHOLD.')
else:
    n_show = min(DISPLAY_N, len(anomaly_indices))
    n_runs_show = min(len(aligned), 4)  # cap at 4 runs per row to fit screen
    cols = 1 + n_runs_show  # GT + one column per run
    class_cmap = mcolors.ListedColormap(CLASS_COLORS + [NODATA_COLOR])
    
    def display_mask(mask):
        m = np.asarray(mask, dtype=np.int16).copy()
        m[m == 255] = 5  # nodata → display slot 5 (gray)
        return m

    for patch_rank, patch_idx in enumerate(anomaly_indices[:n_show]):
        pid = shared_ids_arr[patch_idx]
        fig, axes = plt.subplots(1, cols, figsize=(3 * cols, 3))
        
        # GT
        gt = display_mask(y_true_canonical[patch_idx])
        axes[0].imshow(gt, cmap=class_cmap, vmin=0, vmax=5, interpolation='nearest')
        axes[0].set_title(f'{pid}\nGT')
        axes[0].axis('off')
        
        # Per-run predictions
        for col_i, run in enumerate(aligned[:n_runs_show]):
            pred = display_mask(run['y_pred'][patch_idx])
            pm1 = run['patch_pm1'][patch_idx]
            short = run['run_id'].replace('corrected_unet_pm1bce_', '').replace('corrected_', '')
            short = short[:28]
            axes[col_i + 1].imshow(pred, cmap=class_cmap, vmin=0, vmax=5, interpolation='nearest')
            axes[col_i + 1].set_title(f'{short}\nAcc±1={pm1:.3f}', fontsize=7)
            axes[col_i + 1].axis('off')
        
        fig.suptitle(f'Anomaly #{patch_rank+1} — mean Acc±1={mean_pm1[patch_idx]:.3f}, hard in {hard_fraction[patch_idx]:.0%} of runs', fontsize=9)
        plt.tight_layout()
        plt.show()
    
    # Color legend
    fig, ax = plt.subplots(figsize=(8, 0.6))
    ax.axis('off')
    for i, (name, color) in enumerate(zip(CLASS_NAMES + ['Nodata'], CLASS_COLORS + [NODATA_COLOR])):
        ax.add_patch(plt.Rectangle((i, 0), 0.85, 1, color=color, ec='black', lw=0.5))
        ax.text(i + 0.425, 1.15, name, ha='center', va='bottom', fontsize=8)
    ax.set_xlim(0, 6)
    plt.tight_layout()
    plt.show()

## 7. Diversity — pairwise prediction disagreement

In [ ]:
n_runs = len(aligned)
valid_global = (y_true_canonical >= 0) & (y_true_canonical <= 4)  # (N, H, W)

# Pairwise disagreement rate on valid pixels
disagree = np.zeros((n_runs, n_runs))
for i in range(n_runs):
    for j in range(n_runs):
        if i == j:
            continue
        pred_i = aligned[i]['y_pred'][valid_global]
        pred_j = aligned[j]['y_pred'][valid_global]
        disagree[i, j] = float((pred_i != pred_j).mean())

# Pairwise patch-level Acc±1 correlation
corr = np.corrcoef(pm1_matrix)  # (n_runs, n_runs)

short_names = [r['run_id'].replace('corrected_unet_pm1bce_', '').replace('corrected_', '')[:30] for r in aligned]

if n_runs > 1:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    im0 = axes[0].imshow(disagree, vmin=0, vmax=0.5, cmap='YlOrRd')
    axes[0].set_xticks(range(n_runs)); axes[0].set_xticklabels(short_names, rotation=45, ha='right', fontsize=7)
    axes[0].set_yticks(range(n_runs)); axes[0].set_yticklabels(short_names, fontsize=7)
    axes[0].set_title('Pairwise prediction disagreement rate\n(higher = more diverse)')
    plt.colorbar(im0, ax=axes[0])
    for i in range(n_runs):
        for j in range(n_runs):
            if i != j:
                axes[0].text(j, i, f'{disagree[i,j]:.2f}', ha='center', va='center', fontsize=7)
    
    im1 = axes[1].imshow(corr, vmin=-1, vmax=1, cmap='RdYlGn')
    axes[1].set_xticks(range(n_runs)); axes[1].set_xticklabels(short_names, rotation=45, ha='right', fontsize=7)
    axes[1].set_yticks(range(n_runs)); axes[1].set_yticklabels(short_names, fontsize=7)
    axes[1].set_title('Patch-level Acc±1 correlation\n(lower = more complementary)')
    plt.colorbar(im1, ax=axes[1])
    for i in range(n_runs):
        for j in range(n_runs):
            axes[1].text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center', fontsize=7)
    
    plt.tight_layout()
    plt.show()
else:
    print('Need at least 2 runs for diversity metrics.')

## 8. Ensemble experiments

In [ ]:
def ensemble_probs(aligned_runs: list[dict], weights: np.ndarray | None = None) -> np.ndarray:
    """Weighted average of prob arrays. Returns (N, 5, H, W)."""
    stacked = np.stack([r['probs'] for r in aligned_runs], axis=0)  # (n_runs, N, 5, H, W)
    if weights is None:
        weights = np.ones(len(aligned_runs))
    w = np.array(weights, dtype=np.float32) / np.sum(weights)
    return (stacked * w[:, None, None, None, None]).sum(axis=0)

def eval_ensemble(probs: np.ndarray, y_true: np.ndarray, name: str) -> dict:
    y_pred = probs.argmax(axis=1).astype(np.int16)  # (N, H, W)
    m = global_metrics(y_true, y_pred)
    return {'name': name, **m}

results = []

# Individual runs
for r in aligned:
    short = r['run_id'].replace('corrected_unet_pm1bce_', '').replace('corrected_', '')
    results.append({'name': short, **r['metrics']})

if len(aligned) >= 2:
    # Strategy 1: equal-weight mean probs
    eq_probs = ensemble_probs(aligned)
    results.append(eval_ensemble(eq_probs, y_true_canonical, 'ensemble_equal_weight'))
    
    # Strategy 2: weighted by val Acc±1
    val_pm1_weights = np.array([r['metrics']['acc_pm1'] for r in aligned])
    wt_probs = ensemble_probs(aligned, weights=val_pm1_weights)
    results.append(eval_ensemble(wt_probs, y_true_canonical, 'ensemble_acc_weighted'))
    
    # Strategy 3: majority vote on y_pred
    preds_stacked = np.stack([r['y_pred'] for r in aligned], axis=0)  # (n_runs, N, H, W)
    vote_pred = np.zeros_like(preds_stacked[0])
    valid_mask = (y_true_canonical >= 0) & (y_true_canonical <= 4)
    for c in range(5):
        votes = (preds_stacked == c).sum(axis=0).astype(np.int16)
        # Use argmax later — build a (N, 5, H, W) vote count then argmax
        pass
    vote_counts = np.stack([(preds_stacked == c).sum(axis=0) for c in range(5)], axis=1).astype(np.float32)
    results.append(eval_ensemble(vote_counts, y_true_canonical, 'ensemble_majority_vote'))
    
    # Strategy 4: top-2 by Acc±1 only
    if len(aligned) > 2:
        top2_idx = np.argsort([r['metrics']['acc_pm1'] for r in aligned])[-2:]
        top2 = [aligned[i] for i in top2_idx]
        top2_probs = ensemble_probs(top2)
        results.append(eval_ensemble(top2_probs, y_true_canonical, 'ensemble_top2_equal'))

ens_df = pd.DataFrame(results)
recall_cols = [f'recall_{n}' for n in CLASS_NAMES] if 'per_class_recall' not in ens_df.columns else []

# Flatten per_class_recall
def flatten_row(row):
    if isinstance(row.get('per_class_recall'), dict):
        for c, name in enumerate(CLASS_NAMES):
            row[f'recall_{name}'] = row['per_class_recall'].get(c, np.nan)
        del row['per_class_recall']
    return row

flat_results = [flatten_row(dict(r)) for r in results]
ens_df = pd.DataFrame(flat_results).sort_values('acc_pm1', ascending=False).reset_index(drop=True)
cols_show = ['name', 'acc_pm1', 'exact', 'mae'] + [f'recall_{n}' for n in CLASS_NAMES]
cols_show = [c for c in cols_show if c in ens_df.columns]

display(ens_df[cols_show].style.format({
    'acc_pm1': '{:.4f}', 'exact': '{:.4f}', 'mae': '{:.4f}',
    **{f'recall_{n}': '{:.3f}' for n in CLASS_NAMES}
}).background_gradient(subset=['acc_pm1'], cmap='RdYlGn'))

## 9. Per-class ensemble lift

In [ ]:
if len(aligned) >= 2 and 'eq_probs' in dir():
    best_single = max(aligned, key=lambda r: r['metrics']['acc_pm1'])
    ens_pred = eq_probs.argmax(axis=1).astype(np.int16)
    single_pred = best_single['y_pred']
    
    lift_rows = []
    for c, name in enumerate(CLASS_NAMES):
        class_mask = (y_true_canonical == c)
        n = class_mask.sum()
        if n == 0:
            continue
        single_recall = float((single_pred[class_mask] == c).mean())
        ens_recall = float((ens_pred[class_mask] == c).mean())
        lift_rows.append({'class': name, 'single_best': single_recall, 'ensemble_equal': ens_recall,
                          'lift': ens_recall - single_recall, 'n_pixels': int(n)})
    
    lift_df = pd.DataFrame(lift_rows)
    display(lift_df.style.format({'single_best': '{:.3f}', 'ensemble_equal': '{:.3f}',
                                   'lift': '{:+.3f}', 'n_pixels': '{:,}'}).background_gradient(
        subset=['lift'], cmap='RdYlGn', vmin=-0.1, vmax=0.1))
    
    fig, ax = plt.subplots(figsize=(9, 3.5))
    x = np.arange(len(lift_df))
    w = 0.35
    ax.bar(x - w/2, lift_df['single_best'], width=w, label=f'Best single: {best_single["run_id"].split("_")[2]}', color='steelblue')
    ax.bar(x + w/2, lift_df['ensemble_equal'], width=w, label='Ensemble (equal weight)', color='darkorange')
    ax.set_xticks(x); ax.set_xticklabels(lift_df['class'])
    ax.set_ylabel('Recall')
    ax.set_ylim(0, 1)
    ax.legend()
    ax.set_title('Per-class recall: best single vs equal-weight ensemble')
    plt.tight_layout()
    plt.show()
else:
    print('Need at least 2 corrected runs for lift analysis.')

## 10. Patch-level: ensemble vs best single

In [ ]:
if len(aligned) >= 2 and 'eq_probs' in dir():
    ens_patch_pm1 = patch_acc_pm1(y_true_canonical, eq_probs.argmax(axis=1).astype(np.int16))
    best_single_pm1 = patch_acc_pm1(y_true_canonical, best_single['y_pred'])
    
    delta = ens_patch_pm1 - best_single_pm1  # positive = ensemble better
    wins = (delta > 0.01).sum()
    losses = (delta < -0.01).sum()
    ties = len(delta) - wins - losses
    print(f'Ensemble vs best single — wins: {wins}, losses: {losses}, ties: {ties}')
    print(f'Mean delta: {np.nanmean(delta):+.4f}')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].scatter(best_single_pm1, ens_patch_pm1, s=3, alpha=0.3, color='steelblue')
    lims = [0, 1]
    axes[0].plot(lims, lims, 'k--', lw=0.7, label='no change')
    axes[0].set_xlabel('Best single Acc±1 per patch')
    axes[0].set_ylabel('Ensemble Acc±1 per patch')
    axes[0].set_title('Patch-level: ensemble vs best single')
    axes[0].legend()
    
    axes[1].hist(delta, bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
    axes[1].axvline(0, color='red', lw=0.8, ls='--')
    axes[1].set_xlabel('Acc±1 delta (ensemble − single)')
    axes[1].set_ylabel('Patches')
    axes[1].set_title(f'Distribution of lift per patch (mean {np.nanmean(delta):+.4f})')
    
    plt.tight_layout()
    plt.show()
    
    # Show top patches where ensemble gained most vs single
    gain_idx = np.argsort(delta)[::-1][:5]
    loss_idx = np.argsort(delta)[:5]
    print(f'\nTop ensemble gains:')
    for i in gain_idx:
        print(f'  {shared_ids_arr[i]}  delta={delta[i]:+.3f}  single={best_single_pm1[i]:.3f}  ens={ens_patch_pm1[i]:.3f}')
    print(f'Top ensemble losses:')
    for i in loss_idx:
        print(f'  {shared_ids_arr[i]}  delta={delta[i]:+.3f}  single={best_single_pm1[i]:.3f}  ens={ens_patch_pm1[i]:.3f}')

## 11. Strategy summary

In [ ]:
print('=' * 70)
print('ENSEMBLE STRATEGY SUMMARY')
print('=' * 70)

if ens_df is not None and 'acc_pm1' in ens_df.columns:
    best_row = ens_df.iloc[0]
    print(f"Best strategy:  {best_row['name']}")
    print(f"  Acc±1:  {best_row['acc_pm1']:.4f}")
    print(f"  Exact:  {best_row['exact']:.4f}")
    print(f"  MAE:    {best_row['mae']:.4f}")
    print()

if len(aligned) >= 2:
    print(f'Anomalous patches (hard in ≥{ANOMALY_MIN_RUN_FRACTION:.0%} of runs): {anomaly_mask.sum()}')
    print(f'Partially-hard patches (ensemble opportunity): {partial_mask.sum()}')
    print()
    print('Pairwise disagreement rates (high = more diversity = more ensemble value):')
    for i in range(n_runs):
        for j in range(i+1, n_runs):
            si = aligned[i]['run_id'].replace('corrected_unet_pm1bce_', '')[:30]
            sj = aligned[j]['run_id'].replace('corrected_unet_pm1bce_', '')[:30]
            print(f'  {si} vs {sj}: {disagree[i,j]:.3f}')
    print()

print('Runs available for ensemble (sorted by val Acc±1):')
for r in sorted(aligned, key=lambda x: x['metrics']['acc_pm1'], reverse=True):
    print(f"  {r['metrics']['acc_pm1']:.4f}  {r['run_id']}")

print()
print('Next steps:')
print('  1. Wait for remaining corrected runs (seasonal, concat) to finish')
print('  2. Re-run this notebook once more NPZ files are pulled')
print('  3. Pick top-N ensemble by: (a) high individual Acc±1, (b) high pairwise disagreement')
print('  4. Use equal-weight or acc-weighted probs (majority vote rarely beats prob averaging)')
print('  5. Submit ensemble ZIP only if it beats the 50.63 CodaBench floor')